# ResNet1d ECG benchmark: Lead I vs 12 Leads

Обучаем **чистый 1D ResNet** (без ручных HRV/template фичей) на сырых
ECG-сигналах, для двух вариантов входа:
- `lead1` — только Lead I (1 канал)
- `all` — все 12 отведений

Сплит на train/val/test — **свежий случайный shuffle-сплит с заданными
долями** (fold-колонка из CSV игнорируется). Есть параметр `DATA_FRACTION`
для быстрого бенчмарка на подвыборке данных.

Все параметры вынесены в один конфиг (раздел 1) — эпохи, доли сплита,
DATA_FRACTION, размер модели, батч, lr и т.д.

> ⚠️ **Путь к датасету (`DATA_ROOT`/`PROJECT_DIR`) нужно проверить и
> поправить под своё окружение** — ниже стоит путь-заглушка по аналогии со
> старыми ноутбуками проекта (Colab + Google Drive).

## 0. Монтирование окружения (Colab / Drive)

Пропустите эту ячейку, если работаете не в Colab.

In [12]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Конфиг — все параметры в одном месте

Отредактируйте `PROJECT_DIR` / `DATA_ROOT` под своё окружение — это
единственное, что notebook не может знать сам.

In [13]:
CONFIG = {
    # ---- Пути (ПРОВЕРИТЬ И ПОПРАВИТЬ!) --------------------------------
    'PROJECT_DIR': '/content/drive/MyDrive/Prna/Prna/physionet2020-submission',
    'DATA_ROOT_SUBDIR': 'data/physionet2020',   # относительно PROJECT_DIR
    'RECORDS_CSV': 'records_stratified_10_folds_v2.csv',
    'OUTPUT_DIR': 'resnet_benchmark_results',

    # ---- Воспроизводимость --------------------------------------------
    'SEED': 42,

    # ---- Данные: доля датасета + сплит на train/val/test ---------------
    'DATA_FRACTION': 0.4,      # 0..1, доля записей от всего датасета (после shuffle)
    'MAX_RECORDS': None,       # опционально: жёсткий кап на число записей (или None)
    'TRAIN_FRAC': 0.8,
    'VAL_FRAC': 0.1,
    'TEST_FRAC': 0.1,          # TRAIN_FRAC + VAL_FRAC + TEST_FRAC должны давать 1.0
    'SHUFFLE_SPLIT': True,     # шафлить датасет перед сплитом (fold-колонка игнорируется)

    # ---- Сигнал / окна ---------------------------------------------------
    'WINDOW_SECONDS': 15,
    'NB_WINDOWS_TRAIN': 1,     # окон на запись во время обучения
    'NB_WINDOWS_EVAL': 20,     # окон на запись при вал./тест. инференсе (усредняем прогнозы)

    # ---- Какие варианты входа обучаем ------------------------------------
    'LEADS_VARIANTS': ['all'],   # 'lead1' = 1 отведение, 'all' = 12 отведений

    # ---- Обучение ---------------------------------------------------------
    'N_EPOCHS': 20,
    'PATIENCE': 8,             # early stopping по val AUROC
    'BATCH_SIZE': 32,
    'NUM_WORKERS': 0,
    'LR': 1e-3,

    # ---- Архитектура ResNet1d ----------------------------------------------
    'RESNET_LAYERS': (2, 2, 2, 2),   # число BasicBlock на стадию (ResNet18-style)
    'BASE_PLANES': 64,               # ширина первой стадии (дальше x2, x4, x8)
    'KERNEL_SIZE': 7,
    'DEEPFEAT_SZ': 128,              # размер пулинг-эмбеддинга перед классификатором
    'DROPOUT_RATE': 0.3,

    # ---- Прочее --------------------------------------------------------
    'BETA': 2,                 # для F-beta/G-beta метрик challenge
}

assert abs(CONFIG['TRAIN_FRAC'] + CONFIG['VAL_FRAC'] + CONFIG['TEST_FRAC'] - 1.0) < 1e-6, \
    "TRAIN_FRAC + VAL_FRAC + TEST_FRAC должны суммарно давать 1.0"

CONFIG['DATA_ROOT'] = os.path.join(CONFIG['PROJECT_DIR'], CONFIG['DATA_ROOT_SUBDIR'])

os.chdir(CONFIG['PROJECT_DIR'])
sys.path.insert(0, CONFIG['PROJECT_DIR'])
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

print('PROJECT_DIR:', CONFIG['PROJECT_DIR'])
print('DATA_ROOT  :', CONFIG['DATA_ROOT'], '| exists:', os.path.isdir(CONFIG['DATA_ROOT']))
print(CONFIG)


PROJECT_DIR: /content/drive/MyDrive/Prna/Prna/physionet2020-submission
DATA_ROOT  : /content/drive/MyDrive/Prna/Prna/physionet2020-submission/data/physionet2020 | exists: True
{'PROJECT_DIR': '/content/drive/MyDrive/Prna/Prna/physionet2020-submission', 'DATA_ROOT_SUBDIR': 'data/physionet2020', 'RECORDS_CSV': 'records_stratified_10_folds_v2.csv', 'OUTPUT_DIR': 'resnet_benchmark_results', 'SEED': 42, 'DATA_FRACTION': 0.4, 'MAX_RECORDS': None, 'TRAIN_FRAC': 0.8, 'VAL_FRAC': 0.1, 'TEST_FRAC': 0.1, 'SHUFFLE_SPLIT': True, 'WINDOW_SECONDS': 15, 'NB_WINDOWS_TRAIN': 1, 'NB_WINDOWS_EVAL': 20, 'LEADS_VARIANTS': ['all'], 'N_EPOCHS': 20, 'PATIENCE': 8, 'BATCH_SIZE': 32, 'NUM_WORKERS': 0, 'LR': 0.001, 'RESNET_LAYERS': (2, 2, 2, 2), 'BASE_PLANES': 64, 'KERNEL_SIZE': 7, 'DEEPFEAT_SZ': 128, 'DROPOUT_RATE': 0.3, 'BETA': 2, 'DATA_ROOT': '/content/drive/MyDrive/Prna/Prna/physionet2020-submission/data/physionet2020'}


## 2. Импорты, совместимость, воспроизводимость

In [14]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
)

import ecg_pipeline as ep     # utils проекта: IO, фильтрация, сплиты, метрики
ep.apply_compat_patches()     # патчи для новых numpy/pandas (np.int/np.float/DataFrame.append и т.д.)

random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
torch.manual_seed(CONFIG['SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['SEED'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| gpu count:', torch.cuda.device_count())


device: cuda | gpu count: 1


## 3. Модель: ResNet1d (без ручных HRV/template фичей)

Только сырой сигнал (1 или 12 каналов) + маленький side-input (age, sex).

In [15]:
class BasicBlock1d(nn.Module):
    """Стандартный residual-блок ResNet, 1D-версия."""
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=kernel_size,
                                stride=stride, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=kernel_size,
                                stride=1, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_planes != planes * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm1d(planes * self.expansion),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class ResNet1d(nn.Module):
    """ResNet18-style 1D CNN + side-input (age, sex) fused before the classifier."""

    def __init__(self, classes, in_channels=12, layers=(2, 2, 2, 2),
                 base_planes=64, deepfeat_sz=128, nb_demo=2,
                 kernel_size=15, dropout_rate=0.3):
        super().__init__()
        self.classes = classes
        num_classes = len(classes)

        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_planes, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_planes),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        planes = base_planes
        self.stage1 = self._make_stage(planes, planes, layers[0], stride=1, kernel_size=kernel_size)
        self.stage2 = self._make_stage(planes, planes * 2, layers[1], stride=2, kernel_size=kernel_size)
        self.stage3 = self._make_stage(planes * 2, planes * 4, layers[2], stride=2, kernel_size=kernel_size)
        self.stage4 = self._make_stage(planes * 4, planes * 8, layers[3], stride=2, kernel_size=kernel_size)

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc_embed = nn.Linear(planes * 8, deepfeat_sz)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc_out = nn.Linear(deepfeat_sz + nb_demo, num_classes)

    @staticmethod
    def _make_stage(in_planes, planes, num_blocks, stride, kernel_size):
        blocks = [BasicBlock1d(in_planes, planes, stride=stride, kernel_size=kernel_size)]
        for _ in range(1, num_blocks):
            blocks.append(BasicBlock1d(planes, planes, stride=1, kernel_size=kernel_size))
        return nn.Sequential(*blocks)

    def forward(self, x, wide_feats):
        out = self.stem(x)
        out = self.stage1(out); out = self.stage2(out)
        out = self.stage3(out); out = self.stage4(out)
        out = self.global_pool(out).squeeze(-1)
        out = self.dropout(F.relu(self.fc_embed(out)))
        return self.fc_out(torch.cat([wide_feats, out], dim=1))


def build_resnet1d(classes, leads='all', **kwargs):
    in_channels = 12 if leads == 'all' else 1
    return ResNet1d(classes, in_channels=in_channels, **kwargs)


## 4. Записи: чтение CSV, DATA_FRACTION, shuffle-сплит на train/val/test

In [16]:
data_df, n_total = ep.load_records_table(
    CONFIG['RECORDS_CSV'], CONFIG['DATA_ROOT'],
    data_fraction=CONFIG['DATA_FRACTION'], max_records=CONFIG['MAX_RECORDS'],
    seed=CONFIG['SEED'],
)
print(f"используем {len(data_df)} / {n_total} записей (DATA_FRACTION={CONFIG['DATA_FRACTION']})")


def shuffle_split(df, train_frac, val_frac, test_frac, seed, shuffle=True):
    """Свежий случайный сплит на train/val/test (fold-колонка игнорируется)."""
    if shuffle:
        df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)
    n = len(df)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    trn_df = df.iloc[:n_train].reset_index(drop=True)
    val_df = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    tst_df = df.iloc[n_train + n_val:].reset_index(drop=True)
    return trn_df, val_df, tst_df


trn_df, val_df, tst_df = shuffle_split(
    data_df, CONFIG['TRAIN_FRAC'], CONFIG['VAL_FRAC'], CONFIG['TEST_FRAC'],
    seed=CONFIG['SEED'], shuffle=CONFIG['SHUFFLE_SPLIT'],
)
print('trn:', len(trn_df), 'val:', len(val_df), 'tst:', len(tst_df))


используем 4132 / 10330 записей (DATA_FRACTION=0.4)
trn: 3306 val: 413 tst: 413


## 5. Датасет: только сырой сигнал + age/sex (без HRV/template фичей)

In [17]:
class ECGRawDataset(Dataset):
    def __init__(self, df, window, nb_windows, leads):
        self.df = df.reset_index(drop=True)
        self.window = window
        self.nb_windows = nb_windows
        self.leads = leads  # 'all' | 'lead1'

    def __len__(self):
        return len(self.df)

    def _windows(self, data):
        seq_len = data.shape[-1]
        pad = max(self.window - seq_len, 0)
        if pad > 0:
            data = np.pad(data, ((0, 0), (0, pad + 1)))
            seq_len = data.shape[-1]
        starts = np.random.randint(0, seq_len - self.window + 1, size=self.nb_windows)
        return np.stack([data[:, s:s + self.window] for s in starts])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rec, hdr = ep.load_challenge_data(row.hea_path)

        sig = rec[0:1, :] if self.leads == 'lead1' else rec
        sig = ep.apply_filter(sig, ep.FILTER_BANDWIDTH)
        sig = ep.normalize(sig)
        segs = self._windows(sig).astype(np.float32)

        lbl = row[ep.CLASSES].values.astype(np.float32)
        age = ep.get_age(hdr)
        sex = ep.get_sex(hdr)
        return segs, lbl, age, sex


## 6. Тренировочный / инференс циклы

In [18]:
def make_wide_feats(age_t, sex_t, age_mean, age_std, device):
    age_t = ((age_t.float() - age_mean) / (age_std + 1e-8))[:, None]
    sex_t = sex_t.float()[:, None]
    return torch.cat([age_t, sex_t], dim=1).to(device, non_blocking=True)


def train_one_epoch(model, loader, optimizer, age_mean, age_std, device):
    model.train()
    losses = []

    pbar = tqdm(loader, desc="Training", leave=False)

    for segs, lbl_t, age_t, sex_t in pbar:
        inp = segs[:, 0].to(device, non_blocking=True)   # nb_windows=1 во время трейна
        lbl_t = lbl_t.to(device, non_blocking=True)
        wide = make_wide_feats(age_t, sex_t, age_mean, age_std, device)

        optimizer.zero_grad(set_to_none=True)

        out = model(inp, wide)
        loss = F.binary_cross_entropy_with_logits(out, lbl_t)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{np.mean(losses):.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    return float(np.mean(losses))


def get_probs(model, loader, age_mean, age_std, device):
    model.eval()
    probs_all, lbls_all = [], []
    with torch.no_grad():
        for segs, lbl_t, age_t, sex_t in loader:
            b, nw, c, l = segs.shape
            inp = segs.reshape(b * nw, c, l).to(device, non_blocking=True)
            wide = make_wide_feats(age_t, sex_t, age_mean, age_std, device)
            wide_rep = wide.repeat_interleave(nw, dim=0)
            out = model(inp, wide_rep)
            out = out.view(b, nw, -1).mean(dim=1)
            probs_all.append(out.sigmoid().cpu().numpy())
            lbls_all.append(lbl_t.numpy())
    return np.concatenate(probs_all), np.concatenate(lbls_all)

def compute_extra_metrics(y_true, y_prob, thr=0.5):
    pred = (y_prob >= thr).astype(int)

    return {
        "auprc": average_precision_score(
            y_true, y_prob, average="macro"
        ),
        "f1_macro": f1_score(
            y_true, pred, average="macro", zero_division=0
        ),
        "f1_micro": f1_score(
            y_true, pred, average="micro", zero_division=0
        ),
        "precision": precision_score(
            y_true, pred, average="macro", zero_division=0
        ),
        "recall": recall_score(
            y_true, pred, average="macro", zero_division=0
        ),
        "hamming_loss": hamming_loss(
            y_true, pred
        ),
    }


def make_loader(ds, batch_size, shuffle, num_workers):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=num_workers, pin_memory=True,
                       persistent_workers=num_workers > 0,
                       prefetch_factor=4 if num_workers > 0 else None)


## 7. Пайплайн обучения одного варианта (lead1 / all)

In [19]:
def run_resnet_pipeline(leads, tag, trn_df, val_df, tst_df, window, weights_matrix, config, device):
    trn_ds = ECGRawDataset(trn_df, window, config['NB_WINDOWS_TRAIN'], leads)
    val_ds = ECGRawDataset(val_df, window, config['NB_WINDOWS_EVAL'], leads)
    tst_ds = ECGRawDataset(tst_df, window, config['NB_WINDOWS_EVAL'], leads)

    trn_loader = make_loader(trn_ds, config['BATCH_SIZE'], True, config['NUM_WORKERS'])
    val_loader = make_loader(val_ds, config['BATCH_SIZE'], False, config['NUM_WORKERS'])
    tst_loader = make_loader(tst_ds, config['BATCH_SIZE'], False, config['NUM_WORKERS'])

    age_mean, age_std = float(trn_df.Age.mean()), float(trn_df.Age.std())

    model = build_resnet1d(
        ep.CLASSES, leads=leads,
        layers=config['RESNET_LAYERS'], base_planes=config['BASE_PLANES'],
        kernel_size=config['KERNEL_SIZE'], deepfeat_sz=config['DEEPFEAT_SZ'],
        dropout_rate=config['DROPOUT_RATE'],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{tag}] params: {n_params:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=config['LR'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_auroc, best_state, patience_count = 0., None, 0
    history = []

    for epoch in tqdm(range(CONFIG['N_EPOCHS']), desc="Epochs"):
        trn_loss = train_one_epoch(model, trn_loader, optimizer, age_mean, age_std, device)
        val_probs, val_lbls = get_probs(model, val_loader, age_mean, age_std, device)
        from eval.evaluate_12ECG_score import compute_auc
        val_auroc, _ = compute_auc(val_lbls, val_probs)
        scheduler.step(val_auroc)

        history.append({'epoch': epoch, 'trn_loss': trn_loss, 'val_auroc': val_auroc})
        print(f'[{tag}] epoch {epoch}: trn_loss={trn_loss:.4f} val_auroc={val_auroc:.4f}')

        if val_auroc > best_auroc:
            best_auroc, patience_count = val_auroc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= config['PATIENCE']:
            print(f'[{tag}] early stopping at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_probs, val_lbls = get_probs(model, val_loader, age_mean, age_std, device)

    thrs = ep.find_best_thresholds(val_probs, val_lbls, weights_matrix)

    val_metrics = ep.evaluate_full(
        val_probs,
        val_lbls,
        thrs,
        weights_matrix,
        beta=config["BETA"],
    )

    # Добавляем дополнительные метрики
    val_metrics.update(
        compute_extra_metrics(
            val_lbls,
            val_probs,
            thr=0.5,
        )
    )

    tst_probs, tst_lbls = get_probs(model, tst_loader, age_mean, age_std, device)

    tst_metrics = ep.evaluate_full(
        tst_probs,
        tst_lbls,
        thrs,
        weights_matrix,
        beta=config["BETA"],
    )

    tst_metrics.update(
        compute_extra_metrics(
            tst_lbls,
            tst_probs,
            thr=0.5,
        )
    )

    return {
        'tag': tag, 'leads': leads, 'n_params': n_params, 'thrs': thrs.tolist(),
        'history': pd.DataFrame(history), 'val_metrics': val_metrics, 'tst_metrics': tst_metrics,
    }, model


## 8. Веса классов challenge-метрики

In [20]:
from eval.evaluate_12ECG_score import load_weights
weights_matrix = load_weights('eval/weights.csv', ep.CLASSES)
window = CONFIG['WINDOW_SECONDS'] * ep.FS
print('window (samples):', window)


window (samples): 7500


In [21]:
!pip install -q biosppy peakutils wfdb

## 9. Запуск: обучаем все варианты из `CONFIG['LEADS_VARIANTS']`

По умолчанию — `lead1` и `all` (12 отведений) последовательно на одной GPU.

In [22]:
TAGS = {'lead1': 'ResNet1d (Lead I)', 'all': 'ResNet1d (12 leads)'}

results = {}
models = {}
for leads in CONFIG['LEADS_VARIANTS']:
    tag = TAGS.get(leads, f'ResNet1d ({leads})')
    res, model = run_resnet_pipeline(leads, tag, trn_df, val_df, tst_df, window,
                                      weights_matrix, CONFIG, device)
    results[leads] = res
    models[leads] = model

    ckpt_path = os.path.join(CONFIG['OUTPUT_DIR'], f'resnet1d_{leads}.tar')
    torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG}, ckpt_path)
    print(f"[{tag}] test metrics: {res['tst_metrics']}")


[ResNet1d (12 leads)] params: 8,806,609


Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

Training:   0%|          | 0/104 [00:00<?, ?it/s]

[ResNet1d (12 leads)] epoch 0: trn_loss=0.1198 val_auroc=0.7023


Training:   0%|          | 0/104 [00:00<?, ?it/s]

...

[ResNet1d (12 leads)] epoch 18: trn_loss=0.0381 val_auroc=0.8695


Training:   0%|          | 0/104 [00:00<?, ?it/s]

[ResNet1d (12 leads)] epoch 19: trn_loss=0.0386 val_auroc=0.8460


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

[ResNet1d (12 leads)] test metrics: {'AUROC': np.float64(0.8796519404800063), 'AUPRC': np.float64(0.39499497655657245), 'Fbeta': np.float64(0.3971380240222983), 'Gbeta': np.float64(0.20538227335264284), 'geometric_mean': 0.28559606126218917, 'challenge_metric': 0.6968848058475151, 'auprc': np.float64(0.21944165364254023), 'f1_macro': 0.15162989940608307, 'f1_micro': 0.659919028340081, 'precision': 0.17487836935610315, 'recall': 0.14034700201270614, 'hamming_loss': 0.015065913370998116}


In [23]:
print(res["tst_metrics"])

{'AUROC': np.float64(0.8796519404800063), 'AUPRC': np.float64(0.39499497655657245), 'Fbeta': np.float64(0.3971380240222983), 'Gbeta': np.float64(0.20538227335264284), 'geometric_mean': 0.28559606126218917, 'challenge_metric': 0.6968848058475151, 'auprc': np.float64(0.21944165364254023), 'f1_macro': 0.15162989940608307, 'f1_micro': 0.659919028340081, 'precision': 0.17487836935610315, 'recall': 0.14034700201270614, 'hamming_loss': 0.015065913370998116}
